In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from anthropic import Anthropic 

client = Anthropic()
model = "claude-haiku-5-5"

In [32]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def ask(messages, max_tokens=1024, stop_sequences=None):
    kwargs = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if stop_sequences:
        kwargs["stop_sequences"] = stop_sequences
    return client.messages.create(**kwargs)



def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system
    response = client.messages.create(**params)
    return next(b.text for b in response.content if b.type == "text")



In [28]:
import anthropic, inspect, sys

print("python:", sys.executable)
print("anthropic:", anthropic.__version__, anthropic.__file__)
print("client type:", type(client))
#print("create from:", inspect.getsourcefile(type(client.messages.create)))

python: C:\Users\FARhanKhan\AppData\Local\anaconda3\python.exe
anthropic: 2.12.0 C:\Users\FARhanKhan\AppData\Local\anaconda3\Lib\site-packages\anthropic\__init__.py
client type: <class 'anthropic.Anthropic'>


In [29]:
messages = []
add_user_message(messages, "Generate a one sentence movie idea")
answer = chat(messages, temperature=1.0)
answer

'A retired lighthouse keeper, haunted by a decades-old shipwreck, must team up with a stubborn teenage marine biologist when strange lights begin appearing beneath the waves near the town where her son disappeared.'

In [38]:
#!"C:\Users\FARhanKhan\AppData\Local\anaconda3\python.exe" -m pip install -U anthropic

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.6 MB 3.5 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.6 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 4.3 MB/s  0:00:00
  Attempting uninstall: anthropic
    Found existing installation: anthropic 2.10.0
    Uninstalling anthropic-2.10.0:
      Successfully uninstalled anthropic-2.10.0


In [13]:
messages = []
add_user_message(messages,"Write a 1 sentence description of a fake database")

stream= client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011ChKhph48iMA4fTiUDAJop', container=None, content=[], diagnostics=None, model='claude-haiku-5-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=23, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' fake database is a simulated', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' data', type

In [16]:
messages = []
add_user_message(messages,"Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=100,
    messages=messages

) as stream:
    for text in stream.text_stream:
        #print(text, end="")
        pass

stream.get_final_message()        
    


ParsedMessage(id='msg_011ChKibyzbshvdH7RBSfr6Q', container=None, content=[ParsedTextBlock(citations=None, text='A fake database is a simulated data store filled with invented records, used for testing, development, or demonstrations without exposing real information.', type='text', parsed_output=None)], diagnostics=None, model='claude-haiku-5-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=23, output_tokens=40, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))

In [21]:
messages = []
add_user_message(
    messages,
    "Generate a very short EventBridge rule as JSON. "
    "Return only the JSON object, with no explanation and no code fence."
)
print(chat(messages))

{"Name":"ExampleRule","Description":"Triggers on EC2 instance state changes to running","EventPattern":{"source":["aws.ec2"],"detail-type":["EC2 Instance State-change Notification"],"detail":{"state":["running"]}},"State":"ENABLED"}


In [33]:
# 1. Stop sequence: generation halts when the model emits "</poem>"
messages = []
add_user_message(messages, "Write a short poem about the sea. Wrap it in <poem></poem> tags.")
response = ask(messages, stop_sequences=["</poem>"])

print(response.content[0].text)
if response.stop_reason == "stop_sequence":
    print("Stopped on:", response.stop_sequence)


<poem>
The sea keeps secrets in its blue,
Of ships long sunk and sailors true.
It whispers tales on every wave,
And hums a hymn to shells and cave.

At dawn it glows like hammered gold,
At dusk it hushes, dark and cold,
Yet still it sings its endless song
To shores that wait and hearts that long.

Stopped on: </poem>


In [35]:
# 2. Multiple choice without prefill: ask for a short answer and cap the tokens
messages = []
add_user_message(
    messages,
    "Which is the Greek name for the Sun? (A) Sol (B) Helios (C) Sun. Reply with only the letter.",
)
response = ask(messages, max_tokens=10)
print(response.content[0].text)

B


In [38]:
messages=[]
prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)

text=chat (messages)
text.strip()

from IPython.display import Markdown
Markdown(text)

Here are three short AWS CLI commands:

1. `aws s3 ls`
2. `aws ec2 describe-instances`
3. `aws iam list-users`